# Notebook 02 · España vs Austria

Este notebook reproduce el análisis completo del partido con un enfoque transparente:

- inputs competitivos;
- cálculo de lambdas;
- 100.000 simulaciones;
- probabilidades 1-X-2;
- distribución de goles;
- marcadores más probables;
- análisis de sensibilidad.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if (repo_root / "pmcw").exists():
    sys.path.insert(0, str(repo_root))

from pmcw.poisson import expected_goals
from pmcw import MatchConfig, MatchSimulation

## 1. Inputs `player_based_v1`

Los índices se construyeron a partir de 16 jugadores por selección, ponderados por minutos esperados.

In [ ]:
spain = {
    "team":"España",
    "attack_index":85.33233692344714,
    "midfield_index":87.4633409799784,
    "defense_index":81.94916193513082,
    "goalkeeper_index":89.0,
    "bench_index":74.35568513119534,
    "WCPI":84.45382069027642,
}
austria = {
    "team":"Austria",
    "attack_index":79.15049301708075,
    "midfield_index":80.39665521602174,
    "defense_index":78.28098624382551,
    "goalkeeper_index":81.0,
    "bench_index":69.64235764235765,
    "WCPI":78.54226410348768,
}

inputs_df = pd.DataFrame([spain, austria]).set_index("team").round(2)
inputs_df

In [ ]:
ax = inputs_df.plot(kind="bar", figsize=(11,6))
ax.set_title("Índices competitivos: España vs Austria")
ax.set_xlabel("")
ax.set_ylabel("Rating")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2. Cálculo de goles esperados

La función combina ataque, defensa, medio, WCPI y portería rival.

In [ ]:
lambda_esp = expected_goals(spain, austria)
lambda_aut = expected_goals(austria, spain)

pd.DataFrame({
    "Selección":["España","Austria"],
    "Lambda":[lambda_esp, lambda_aut]
})

## 3. Simulación Monte Carlo

Se generan 100.000 marcadores independientes usando distribuciones de Poisson.

In [ ]:
config = MatchConfig(
    home_team="España",
    away_team="Austria",
    lambda_home=lambda_esp,
    lambda_away=lambda_aut,
    n_simulations=100_000,
    seed=20260702,
)

result = MatchSimulation(config).run().summary()
result

## 4. Probabilidades en 90 minutos

In [ ]:
outcome_df = pd.DataFrame({
    "Resultado":["España gana","Empate","Austria gana"],
    "Probabilidad":[result["home_win"], result["draw"], result["away_win"]]
})
outcome_df["Porcentaje"] = outcome_df["Probabilidad"] * 100
outcome_df

In [ ]:
ax = outcome_df.plot(
    x="Resultado", y="Porcentaje", kind="bar",
    legend=False, figsize=(8,5)
)
ax.set_title("Probabilidades 1-X-2")
ax.set_ylabel("Probabilidad (%)")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Distribución simulada de goles

In [ ]:
rng = np.random.default_rng(20260702)
goals_esp = rng.poisson(lambda_esp, 100_000)
goals_aut = rng.poisson(lambda_aut, 100_000)

dist_esp = pd.Series(goals_esp).value_counts(normalize=True).sort_index()
dist_aut = pd.Series(goals_aut).value_counts(normalize=True).sort_index()

max_goal = max(dist_esp.index.max(), dist_aut.index.max())
goal_distribution = pd.DataFrame({
    "Goles":range(max_goal + 1),
    "España":[dist_esp.get(i,0) for i in range(max_goal + 1)],
    "Austria":[dist_aut.get(i,0) for i in range(max_goal + 1)],
})
goal_distribution.head(8)

In [ ]:
ax = goal_distribution.set_index("Goles").plot(kind="bar", figsize=(10,5))
ax.set_title("Distribución de goles")
ax.set_ylabel("Probabilidad")
ax.set_xlabel("Número de goles")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Marcadores más frecuentes

In [ ]:
scorelines_df = pd.DataFrame(result["top_scorelines"])
scorelines_df["Porcentaje"] = scorelines_df["probability"] * 100
scorelines_df

In [ ]:
ax = scorelines_df.head(10).plot(
    x="scoreline", y="Porcentaje", kind="bar",
    legend=False, figsize=(10,5)
)
ax.set_title("Top 10 marcadores más probables")
ax.set_ylabel("Probabilidad (%)")
ax.set_xlabel("Marcador")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Análisis de sensibilidad

Se modifica el lambda de España para comprobar cuánto dependen los resultados de ese input.

In [ ]:
rows = []
for delta in np.linspace(-0.30, 0.30, 13):
    cfg = MatchConfig(
        home_team="España",
        away_team="Austria",
        lambda_home=lambda_esp + delta,
        lambda_away=lambda_aut,
        n_simulations=30_000,
        seed=20260702,
    )
    r = MatchSimulation(cfg).run().summary()
    rows.append({
        "lambda_esp":lambda_esp + delta,
        "esp_win":r["home_win"],
        "draw":r["draw"],
        "aut_win":r["away_win"],
    })

sensitivity_df = pd.DataFrame(rows)
sensitivity_df.head()

In [ ]:
ax = sensitivity_df.plot(
    x="lambda_esp",
    y=["esp_win","draw","aut_win"],
    figsize=(10,5)
)
ax.set_title("Sensibilidad al lambda de España")
ax.set_xlabel("Lambda de España")
ax.set_ylabel("Probabilidad")
plt.tight_layout()
plt.show()

## 8. Conclusiones

- España aparece como favorita, pero sin dominio absoluto.
- El empate tiene un peso relevante.
- Los marcadores más probables son cortos.
- El análisis de sensibilidad muestra que el resultado depende de los inputs.
- El modelo debe comunicarse como distribución de escenarios, no como certeza.